In [25]:
import os
from  pathlib import Path
import sys
# Установка базовой директории и пути к файлу с учетными данными. Используем конструкцию try-except для обработки возможных ошибок при определении пути для notebook.
try:
    BASE_DIR = Path(__file__).resolve().parents[2]
except NameError:
    BASE_DIR = Path.cwd().resolve().parents[1] 

SRC_DIR = os.path.join(BASE_DIR, 'src')

# Добавляем src в пути поиска модулей
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)


from dotenv import load_dotenv
load_dotenv()

import os
import asyncio
from openai import AsyncOpenAI, OpenAI
import json

python-dotenv could not parse statement starting at line 13


In [11]:
# config.py

# Получаем токен для работы с chst gpt
OPENAI_API_KEY =os.getenv("OPENAI_API_KEY")
# Выбираем модель нейросети
MODEL_NAME = "gpt-4o-mini"
# Выбрасываем ошибку, если не найден токен
if not OPENAI_API_KEY: raise ValueError("OPENAI_API_KEY is not set in environment variables")

In [ ]:
# openai_client.py

client = OpenAI(api_key=OPENAI_API_KEY)
def ask_llm(messages: list[dict]) -> str:
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=0.3
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"LLM Error: {str(e)}"

In [29]:
# prompts.py

def build_wb_analysis_messages(news: str) -> list[dict]:
    answer_structure = {
    "summary": "",
    "financial_risks": [],
    "operational_risks": [],
    "reputation_risks": [],
    "opportunities": [],
    "priority_actions": [],
    "impact_score": 0
    }
    system_prompt = f"""
    Ты — эксперт по торговле на Wildberries.

    Проанализируй новость и верни ответ строго в формате JSON.

    Требования:
    - Только JSON
    - Без markdown
    - Без пояснений
    - Используй двойные кавычки
    - Не добавляй текст вне JSON

    Структура ответа:

    {answer_structure}
    """ 
    user_prompt = f"""
    Контекст:
    - Я селлер на WB
    - Работаю по FBO/FBS
    - Для меня важны: оборачиваемость, остатки, out-of-stock, % выкупа
    - Я анализирую складские остатки и планирование поставок

    Новость:
    {news}

    Проанализируй:
    - повлияет ли это на управление остатками?
    - даёт ли это стратегическое преимущество?
    - нужно ли менять процессы?
    - влияет ли это на прогнозирование оборачиваемости?
    - влияет ли это на частоту поставок?
    - можно ли автоматизировать аналитику?

    impact_score:
        1–3 = информационное обновление
        4–6 = требует анализа
        7–10 = серьёзно влияет на юнит-экономику

    - Если данных недостаточно — оставь пустые массивы
    - impact_score должен быть числом от 1 до 10. 
    Жёсткое правило:
    - impact_score не может быть выше 5, если новость не изменяет напрямую:
    - комиссии
    - штрафы
    - видимость
    - логистику
    - Если новость только добавляет новый отчёт, impact_score = 3

    financial_risks = изменения комиссий, штрафов, ценообразования
    operational_risks = ошибки складов, доставка, out-of-stock
    reputation_risks = рейтинг, отзывы, видимость
"""

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

In [28]:
# notebook
news = "Новый отчёт по истории остатков в CSV. Добавили отчёт CSV модели InventoryHistoryReportRes. В запросе на создание отчёта reportType этой модели — STOCK_HISTORY_DAILY_CSV.В отчёте показываются данные об остатках по состоянию на 23:59. Данный отчёт доступен без подписки Джем."
messages = build_wb_analysis_messages(news)
response = ask_llm(messages)


try:
    parsed = json.loads(response)
    print(parsed)
except json.JSONDecodeError as e:
    print("Ошибка JSON:", e)
print(response)

{'summary': 'Добавление нового отчёта по истории остатков в CSV может улучшить управление остатками и прогнозирование, но не требует изменения процессов.', 'financial_risks': [], 'operational_risks': [], 'reputation_risks': [], 'opportunities': [], 'priority_actions': [], 'impact_score': 3}
{
    "summary": "Добавление нового отчёта по истории остатков в CSV может улучшить управление остатками и прогнозирование, но не требует изменения процессов.",
    "financial_risks": [],
    "operational_risks": [],
    "reputation_risks": [],
    "opportunities": [],
    "priority_actions": [],
    "impact_score": 3
}


In [30]:
response

'{\n    "summary": "Добавление нового отчёта по истории остатков в CSV может улучшить управление остатками и прогнозирование, но не требует изменения процессов.",\n    "financial_risks": [],\n    "operational_risks": [],\n    "reputation_risks": [],\n    "opportunities": [],\n    "priority_actions": [],\n    "impact_score": 3\n}'